# Open Text — extract by resource ID

Run **`@istari:extract`** on a Model you already uploaded in the Istari Digital web app.

You will:

1. Connect with a personal access token
2. Paste the Model UUID
3. Submit `@istari:extract` and wait for artifacts

See [Open Text](https://docs.istaridigital.com/integrations/Productivity/open_text) for supported types and the SDK sample this notebook follows.

### Prerequisites

- From the cookbook root: `uv sync --group dev` (kernel **Python (istari-client-cookbook)**).
- [`samples/.env`](../../.env) with `ISTARI_REGISTRY_URL` and `ISTARI_PERSONAL_ACCESS_TOKEN`.
- An agent with **`textract`** / **`@istari:extract`**.

> **Before you run the job:** in the web app, upload a file under **Files** (for example [`sample.txt`](./sample.txt) in this folder) and copy its Model UUID into **Prep**.

## 1 · Connect

In [ ]:
import os
from time import sleep

import dotenv
from istari_digital_client import Client, Configuration, JobStatusName

# samples/.env — this notebook lives in samples/connect/opentext/
dotenv.load_dotenv("../../.env")

registry_url = os.environ["ISTARI_REGISTRY_URL"]
token = os.environ["ISTARI_PERSONAL_ACCESS_TOKEN"]

client = Client(Configuration(registry_url=registry_url, registry_auth_token=token))
me = client.get_current_user()
print(f"Signed in as {me.display_name} ({me.email})")

## 2 · Prep

Set `RESOURCE_ID` to the Model UUID from **Files**. Set `OPERATING_SYSTEM` to the OS of the agent that will run the job.

In [ ]:
# Upload the file in the Istari Digital web app first, then paste the Model UUID here.
RESOURCE_ID = ""

TOOL_NAME = "textract"
TOOL_VERSION = "1.6.3"  # tool version from the Open Text docs
OPERATING_SYSTEM = "Windows 11"  # must match the agent (e.g. Ubuntu 22.04, RHEL 8)

print(RESOURCE_ID, TOOL_NAME, TOOL_VERSION, OPERATING_SYSTEM)

## 3 · Submit `@istari:extract`

`add_job` starts the work on the agent. The loop below calls `get_job` until status is terminal. Expected artifacts: `extracted_text.txt` and `metadata_report.json`.

In [ ]:
model = client.get_model(RESOURCE_ID)

job = client.add_job(
    model_id=model.id,
    function="@istari:extract",
    tool_name=TOOL_NAME,
    tool_version=TOOL_VERSION,
    operating_system=OPERATING_SYSTEM,
)
print("Submitted job", job.id)

# Refresh until the agent finishes (or fails / is canceled).
done = (JobStatusName.COMPLETED, JobStatusName.FAILED, JobStatusName.CANCELED)
while job.status.name not in done:
    sleep(5)
    job = client.get_job(job.id)
    print(job.status.name.value)

if job.status.name != JobStatusName.COMPLETED:
    raise RuntimeError(f"Job {job.id} ended with {job.status.name.value}: {job.status.message}")

# Reload so we see artifacts written by this job.
model = client.get_model(model.id)
for artifact in model.artifacts or []:
    print(artifact.name)

## Learn more

- [Open Text](https://docs.istaridigital.com/integrations/Productivity/open_text)
- [Python Client — Quick Start](https://docs.istaridigital.com/developers/SDK/setup)